In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

**<font size="6" color="red">ch3. 연관분석</font>**
- pip install apyori

# 1. 연관분석 개요
- 데이터들 사이에 자주 발생하는 속성을 찾고, 그 속성들 사이에 연관성이 어느 정도 있는지 분석
- 활용분야 : 이벤트미리감지(사기적발..), 신상품카테고리 구성

[조건:left-hand side:오렌지주소]->[결과:right-hand side:와인]
```
- 연과분석과 관련된 지표
1. 지지도(support) : 얼마나 자주 함께 나타나는지
    (lhs, rhs)의 항목수/전체항목수 = 0.2
    
2. 신뢰도(confidence) : 조건이 오면 결과가 얼마나 자주 나타나는지
    (lhs->rhs)의 항목수/lhs의 항목수 = 1/2 = 0.5
    
3. 향상도(lift) : 우연히 발생한 규칙은 아닌지 확인
    lhs->rhs의 지지도 / (lhs의 지지도*rhs의 지지도)
    => 0.2 / (0.4*0.6) = 0.2/0.24 = 0.833
    향상도<1 : 기대가 낮다
    향상도>1 : 기대가 높다
```

# 2. 연관분석 구현

In [ ]:
import csv
with open('data/cf_basket.csv', 'r', encoding='utf-8') as f:
    csvdata = csv.reader(f)
    # print(list(csvdata))
    transaction = list(csvdata)
transaction

In [ ]:
from apyori import apriori
rules = apriori(transaction, # 2차원 데이터
               min_support=0.15,
               min_confidence=0.1)
rules = list(rules)
len(rules)

In [ ]:
rules[10]

In [ ]:
rule = rules[10]
support = rule[1]
order_st = rule[2]
for item in order_st:
    lhs = item[0]
    rhs = item[1]
    confidence = item[2]
    lift = item[3]
    if lift > 1:
        print("{}=>{}\t {}\t {}\t {}".format(lhs, rhs, support, 
                                             round(confidence,2), 
                                             round(lift,2)))

In [ ]:
for rule in rules:
    support = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = ', '.join([data for data in item[0]])
        rhs = ', '.join([data for data in item[1]])
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            print("{}=>{}\t {}\t {}\t {}".format(lhs, rhs, support, 
                                                 round(confidence,2), 
                                                 round(lift,2)))

In [ ]:
import pandas as pd
rules_df = pd.DataFrame(None, columns=['lhs', 'rhs', '지지도', '신뢰도', '향상도'])
# rules_df.loc[0] = ['와인', '오렌지', 0.15, 0.5, 1.1] 식으로 for문내에서 데이터추가
idx = 0

for rule in rules:
    support = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = ', '.join([data for data in item[0]])
        rhs = ', '.join([data for data in item[1]])
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            rules_df.loc[idx] = [lhs, rhs, support, round(confidence,2), round(lift,2)]
            idx += 1
rules_df.sort_values(by=['향상도', '신뢰도'], ascending=False)

# 3. 경주/전주 여행 자료 연관분석

In [ ]:
import pandas as pd
from konlpy.tag import Hannanum, Komoran, Kkma
df = pd.read_csv('data/naver_kin.csv', sep='\t')
total_text_list = df['total_text'].to_list()
analyzer = Hannanum()
total_noun_list = []
select_pos = ['NC', 'NQ'] # 보통명사, 고유명사
불용어 = {'여행', '전주여행', '경주여행'}
for total_text in total_text_list[:5]:
    # total_noun = analyzer.nouns(total_text)
    total_noun = [token for token, tag in analyzer.pos(total_text, ntags=22)
                    if tag in select_pos and
                    token not in 불용어 and
                    len(token)>1]
    total_noun_list.append(total_noun)
print(total_noun_list[:2])

In [ ]:
import pandas as pd
from konlpy.tag import Hannanum, Kkma, Komoran
df = pd.read_csv('data/naver_kin.csv', sep='\t')
total_text_list = df['total_text'].to_list()
# total_text_list[:2]
analyzer = Komoran()
total_noun_list = []
select_pos = ['NC','NQ'] # Hannanum 보통명사, 고유명사
select_pos = ['NNP', 'NNG']# Kkma, Komoran 보통명사, 고유명사
불용어 = {'여행'}
for total_text in total_text_list:
    #total_noun = analyzer.nouns(total_text)
    total_noun = [token for token, tag in analyzer.pos(total_text)
                     if tag in select_pos and
                         token not in 불용어 and
                         len(token)>1]
    total_noun_list.append(total_noun)
print(total_noun_list[:2])

In [ ]:
%%time
rules = apriori(total_noun_list, min_support=0.15, min_confidence=0.3)
rules = list(rules)
len(rules)

In [ ]:
rules_df = pd.DataFrame(None, columns=['lhs','rhs','지지도','신뢰도','향상도'])
#rules_df.loc[0] = ['와인','오렌지', 0.15, 0.5, 1.1] 식으로 for문내에서 데이터추가
idx = 0
for rule in rules:
    support = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = ', '.join([data for data in item[0]])
        rhs = ', '.join([data for data in item[1]])
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            rules_df.loc[idx] = [lhs, rhs, support, round(confidence, 2), round(lift,2)]
            idx += 1
rules_df.sort_values(by=['향상도','신뢰도'], ascending=False, inplace=True)
rules_df = rules_df.reset_index(drop=True)

In [ ]:
pd.options.display.max_rows

In [ ]:
rules_df.head(60)